In [17]:
import sys
import os
sys.path.append(os.path.abspath('..'))


import pandas as pd
import numpy as np

from data_preparation.data_processor import DataProcessor
import matplotlib.pyplot as plt
from darts import TimeSeries

from darts.models import RNNModel, RegressionModel
from darts.metrics import mape
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


In [18]:
waste_data = pd.read_csv("../synthetic_waste_data.csv")

fetcher = DataProcessor(waste_data)
unique_companies = fetcher.waste_data['waste_type'].unique()
company_dfs = {}


for company in unique_companies:
    prep_data_company = fetcher.agg_quantity(waste_type=company, by_waste_type= True).reset_index()
    prep_data_company["quantity_tons"] = prep_data_company["quantity_tons"].diff()
    prep_data_company = prep_data_company.dropna()
    company_dfs[company] = TimeSeries.from_dataframe(prep_data_company, 'date', 'quantity_tons')

In [19]:
company_dfs["Municipal"]

<TimeSeries (DataArray) (date: 1095, component: 1, sample: 1)> Size: 4kB
array([[[-1.6381126e+01]],

       [[ 2.3026363e+02]],

       [[-1.0619979e+02]],

       ...,

       [[-8.4930397e+01]],

       [[ 5.6576164e+01]],

       [[ 1.6978073e-01]]], dtype=float32)
Coordinates:
  * date       (date) datetime64[ns] 9kB 2022-01-02 2022-01-03 ... 2024-12-31
  * component  (component) object 8B 'quantity_tons'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  None
    hierarchy:          None

In [20]:
# Define model creation functions to ensure fresh instances for each company
def create_gru_model():
    return RNNModel(
        model='GRU', 
        input_chunk_length=12, 
        output_chunk_length=1,
        hidden_dim=32, 
        n_rnn_layers=1, 
        dropout=0.2, 
        batch_size=16, 
        n_epochs=100, 
        optimizer_kwargs={'lr': 1e-3}, 
        random_state=42
    )

def create_xgboost_model():
    return RegressionModel(
        model=XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
        lags=12,
        output_chunk_length=1
    )

def create_catboost_model():
    return RegressionModel(
        model=CatBoostRegressor(iterations=100, learning_rate=0.1, random_state=42),
        lags=12,
        output_chunk_length=1
    )

# Define the model creation functions dictionary
model_creators = {
    'GRU': create_gru_model,
#    'XGBoost': create_xgboost_model,
#    'CatBoost': create_catboost_model
}

# Initialize dictionaries to store the results
mape_results = {model_name: [] for model_name in model_creators}
mae_results = {model_name: [] for model_name in model_creators}
rmse_results = {model_name: [] for model_name in model_creators}

# Iterate over each company
for company in company_dfs:
    print(f"Processing company: {company}")
    series = company_dfs[company]
    
    # Split the data into training and validation sets
    train_size = int(len(series) * 0.8)
    train, val = series[:train_size], series[train_size:]
    
    # Iterate over each model
    for model_name, model_creator in model_creators.items():
        print(f"Training {model_name} for {company}")
        try:
            # Create a fresh instance of the model
            model = model_creator()
            
            # Train the model
            model.fit(train)
            
            # Make predictions on the validation set
            preds = model.predict(len(val))
            
            # Calculate metrics
            #mape_score = mape(val, preds)
            #mae_score = mean_absolute_error(val.values(), preds.values())
            rmse_score = root_mean_squared_error(val.values(), preds.values())
            
            # Store the results
            #mape_results[model_name].append(mape_score)
            #mae_results[model_name].append(mae_score)
            rmse_results[model_name].append(rmse_score)
            
            print(f"{model_name} for {company} , RMSE: {rmse_score:.4f}")
            
        except Exception as e:
            print(f"Error training {model_name} for {company}: {str(e)}")



ignoring user defined `output_chunk_length`. RNNModel uses a fixed `output_chunk_length=1`.
c:\Users\maxik\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | rnn             | GRU              | 3.4 K  | train
6 | V               | Linear           | 33     | trai

Processing company: Municipal
Training GRU for Municipal


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

...

GRU for Construction , RMSE: 47.5319
Processing company: Commercial
Training GRU for Commercial


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

GRU for Commercial , RMSE: 54.3702


In [21]:
import math
# Calculate the average metrics for each model across all companies

avg_rmse_results = {model_name: sum(scores) / len(scores) if scores else float('nan') for model_name, scores in rmse_results.items()}


print("\nAverage RMSE scores:")
for model_name, score in avg_rmse_results.items():
    if not math.isnan(score):
        print(f"{model_name}: {score:.4f}")
    else:
        print(f"{model_name}: No valid results")


Average RMSE scores:
GRU: 65.6743


In [22]:
# Filter only GRU results for different waste types
gru_waste_results = {model_name: scores for model_name, scores in rmse_results.items() if 'GRU' in model_name}

print(gru_waste_results)

{'GRU': [100.32325, 72.46542, 53.680504, 47.531853, 54.370235]}
